In [1]:
from macro import *
from bao import *
from static import *

In [2]:
with open(PATH_MACRO_DICTIONARY, "r") as a:
    macro_dictionary = json.load(a)

In [3]:
macro_dictionary

{'URL_IMF': 'http://dataservices.imf.org/REST/SDMX_JSON.svc/',
 'DATA_KEY': {'CPI': 'CompactData/IFS/M..PCPI_IX',
  'EXCHANGE_RATE': 'CompactData/IFS/M..PCPI_IX',
  'EXPORT': 'CompactData/IFS/M..TXG_FOB_USD',
  'IMPORT': 'CompactData/IFS/M..TMG_CIF_USD',
  'INTEREST_LENDING': 'CompactData/IFS/M.VN.FILR_PA',
  'INTEREST_BENCH_MARK': 'CompactData/IFS/M.VN.FPOLM_PA',
  'INTEREST_DEPOSIT': 'CompactData/IFS/M.VN.FIDR_PA'},
 'CPI_PARTITION': 'country_name'}

In [4]:
macro_dictionary["URL_IMF"]

'http://dataservices.imf.org/REST/SDMX_JSON.svc/'

In [ ]:
macro_data = MacroData("./data/macro")

In [2]:
df = macro_data.cpi()

There is some data in the given path, need to check, wait for results
/cpi: no conflict in the new data
/cpi: no new data


In [5]:
def get_imf(key, additional_col=None):
    data = requests.get(f"{url}{key}").json()["CompactData"][
        "DataSet"
    ]["Series"]
    all_data = []
    for dataset in data:
        observations = dataset.get("Obs", [])
        country_name = dataset.get("@REF_AREA", None)
        try:
            temp_df = pd.DataFrame(observations)
        except ValueError:
            continue
        temp_df.rename(
            columns={
                "@TIME_PERIOD": "TIME_PERIOD",
                "@OBS_VALUE": "value",
            },
            inplace=True,
        )
        temp_df["country_name"] = country_name
        try:
            temp_df["value"] = pd.to_numeric(temp_df["value"])
        except KeyError:
            continue
        if additional_col is not None:
            for col in additional_col.items():
                temp_df[col[1]] = dataset.get(col[0], None)
        all_data.append(temp_df)
    final_df = pd.concat(all_data, ignore_index=True)
    return final_df

In [7]:
url: str = "http://dataservices.imf.org/REST/SDMX_JSON.svc/"

In [8]:
# test CPI
get_imf(
    "CompactData/IFS/M..PCPI_IX", {"@BASE_YEAR": "base_year"}
).head(2)

,TIME_PERIOD,value,@OBS_STATUS,country_name,base_year
0,1975-07,36.703989,NaN,BH,2010=100
1,1975-08,38.906228,NaN,BH,2010=100


In [9]:
# test exchange
get_imf("CompactData/IFS/M..ENDE_XDC_USD_RATE").head(2)

,TIME_PERIOD,value,@OBS_STATUS,country_name
0,1957-01,0.000544,NaN,CL
1,1957-02,0.000564,NaN,CL


In [10]:
# test Export
get_imf("CompactData/IFS/M..TXG_FOB_USD").head(2)

,TIME_PERIOD,value,country_name
0,1950-01,404.221969,XS25
1,1950-02,404.221969,XS25


In [11]:
# test import
get_imf("CompactData/IFS/M..TMG_CIF_USD").head(2)

,TIME_PERIOD,value,country_name
0,1950-01,335.09272,XS25
1,1950-02,335.09272,XS25


In [ ]:
interest_rate = [
    "FILR_PA",  
    
    """ Financial, Interest rates, Lending Rate, Percent per annum
    This refers to the average lending rate charged by banks on loans to the private sector. It's a key indicator of the cost of borrowing money in an economy"""
    "FPOLM_PA",  
    """ 
    Financial, Interest Rates, Monetary Policy-Related Interest Rate, Percent per annum
    This represents the policy rate set by a country's central bank. It's the benchmark interest rate that influences other interest rates in the economy and is used to manage inflation and economic growth.
    """
    "FIDR_PA",  
    """_summary_
    
    # Financial, Interest Rates, Deposit, Percent per annum
    This is the average interest rate paid by banks on deposits from the private sector. It reflects the return individuals and businesses receive for keeping their money in bank accounts.
    """
]
data1 = pd.DataFrame()
data2 = pd.DataFrame()
data3 = pd.DataFrame()
data0 = [data1, data2, data3]
for i, j in zip(interest_rate, data0):
    # Get exchange rate country
    url = "http://dataservices.imf.org/REST/SDMX_JSON.svc/"
    key = "CompactData/IFS/M.VN.%s" % (i)
    data = requests.get(f"{url}{key}").json()["CompactData"][
        "DataSet"
    ]["Series"]
    # yr=data['@BASE_YEAR']
    data_list = [
        [obs.get("@TIME_PERIOD"), obs.get("@OBS_VALUE")]
        for obs in data["Obs"]
    ]
    df = pd.DataFrame(data_list, columns=["date", "value"])
    df = df.set_index(pd.to_datetime(df["date"]))["value"].astype(
        "float"
    )


In [18]:
url = "http://dataservices.imf.org/REST/SDMX_JSON.svc/"
key = "Dataflow"  # Method with series information
search_term = "IFS"  # Term to find in series names
series_list = requests.get(f"{url}{key}").json()["Structure"][
    "Dataflows"
]["Dataflow"]
# Use dict keys to navigate through results:
for series in series_list:
    if search_term in series["Name"]["#text"]:
        print(
            f"{series['Name']['#text']}: {series['KeyFamilyRef']['KeyFamilyID']}"
        )

International Financial Statistics (IFS), 2019 M01: IFS_2019M01
International Financial Statistics (IFS), 2019 M02: IFS_2019M02
International Financial Statistics (IFS), 2018 M03: IFS_2018M03
International Financial Statistics (IFS), 2018 M08: IFS_2018M08
International Financial Statistics (IFS), 2017 M08: IFS_2017M08
International Financial Statistics (IFS), 2019 M05: IFS_2019M05
International Financial Statistics (IFS), 2020 M05: IFS_2020M05
International Financial Statistics (IFS), 2019 M07: IFS_2019M07
International Financial Statistics (IFS), 2018 M12: IFS_2018M12
International Financial Statistics (IFS), 2020 M01: IFS_2020M01
International Financial Statistics (IFS), 2019 M04: IFS_2019M04
International Financial Statistics (IFS), 2019 M09: IFS_2019M09
International Financial Statistics (IFS), 2018 M06: IFS_2018M06
International Financial Statistics (IFS), 2017 M12: IFS_2017M12
International Financial Statistics (IFS), 2020 M07: IFS_2020M07
International Financial Statistics (IFS)

In [ ]:
key = "DataStructure/IFS"  # Method / series
dimension_list = requests.get(f"{url}{key}").json()["Structure"][
    "KeyFamilies"
]["KeyFamily"]["Components"]["Dimension"]
for n, dimension in enumerate(dimension_list):
    print(f"Dimension {n+1}: {dimension['@codelist']}")
key = f"CodeList/{dimension_list[2]['@codelist']}"
code_list = requests.get(f"{url}{key}").json()["Structure"][
    "CodeLists"
]["CodeList"]["Code"]
for code in code_list:
    if "Export" in code["Description"]["#text"]:
        print(f"{code['Description']['#text']}: {code['@value']}")
    # print(f"{code['Description']['#text']}: {code['@value']}")

Dimension 1: CL_FREQ
Dimension 2: CL_AREA_IFS
Dimension 3: CL_INDICATOR_IFS
Exports of Goods and Services, Nominal, Domestic Currency: NX_XDC
Exports of Goods and Services, Nominal, Seasonally Adjusted, Domestic Currency: NX_SA_XDC
Exports of Goods and Services, Nominal, Unadjusted, Domestic Currency: NX_NSA_XDC
Exports of Goods and Services, Real, Domestic Currency: NX_R_XDC
Exports of Goods and Services, Real, Seasonally Adjusted, Domestic Currency: NX_R_SA_XDC
Exports of Goods and Services, Real, Unadjusted, Domestic Currency: NX_R_NSA_XDC
Exports of Goods, Nominal, Domestic Currency: NXG_XDC
Exports of Goods, Nominal, Seasonally Adjusted, Domestic Currency: NXG_SA_XDC
Exports of Goods, Nominal, Unadjusted, Domestic Currency: NXG_NSA_XDC
Exports of Goods, Real, Domestic Currency: NXG_R_XDC
Exports of Goods, Real, Seasonally Adjusted, Domestic Currency: NXG_R_SA_XDC
Exports of Goods, Real, Unadjusted, Domestic Currency: NXG_R_NSA_XDC
Exports of Services, Nominal, Domestic Currency: N